<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [1]:
import sys
import os
sys.path.append(os.path.abspath("../../.."))

from config.spark_config import SparkConfig
from utils.logger import LoggerFactory
from config.io_config import *
from app.platform_app import PlatformApp
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [2]:
# Initialize shared logger (all logs in this run go to the same file: etl_<run_id>.log)
logger = LoggerFactory.setup_logger(name="ETL", log_dir=LOG_DIR)

# Create Spark session with logging enabled (for tracing Spark-related operations)
spark = SparkConfig.create_spark(app_name="Paypal Analytic", logger=logger, use_databricks=True)

# Initialize main application with Spark and logger (used across ETL pipeline)
app = PlatformApp(spark=spark, logger=logger, catalog_name="paypal_analytic")

2026-04-08 22:17:21 | INFO     | ETL | logger.py:113 | Logger initialized | level=DEBUG | file=C:/01_Data/05-data-engineer-bootcamp/03_paypal_databricks/logs\etl_20260401_193104_713773.log
2026-04-08 22:17:23 | INFO     | ETL | spark_config.py:89 | Connected to Databricks via Spark Connect.
2026-04-08 22:17:23 | INFO     | ETL | platform_app.py:44 | Initializing Data Platform...
2026-04-08 22:17:23 | INFO     | ETL | platform_app.py:50 | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [3]:
df_bronze_disputes = spark.sql(f"SELECT * FROM {BRONZE_DISPUTE_TRANSACTIONS}")

# Preview result
df_bronze_disputes.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------+--------+------------------------

## Transformations

### Select Features

In [4]:
df_silver_disputes_transactions = df_bronze_disputes.select("dispute_id", "create_time", "update_time", "adjudications", 
                                                            "elton_created_at", "dt", "hour")

# Preview result
df_silver_disputes_transactions.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+
|dispute_id        |create_time                   |update_time                   |adjudications                                                                                                                                                                                                                                                                                                                        |elton_created_at              |dt      |hour|
+------------------+------------------------------+------------------------------+----------

### Check NULL

In [5]:
# check null
check_null(df = df_silver_disputes_transactions, logger=logger)

2026-04-08 22:17:55 | INFO     | ETL | data_quality.py:55 | No missing values detected in 254 rows.


### Trim spaces

In [6]:
# Remove those trim values
df_silver_disputes_transactions = clean_dataframe(df=df_silver_disputes_transactions)

# Preview result
df_silver_disputes_transactions.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+
|dispute_id        |create_time                   |update_time                   |adjudications                                                                                                                                                                                                                                                                                                                        |elton_created_at              |dt      |hour|
+------------------+------------------------------+------------------------------+----------

### Duplicates

In [7]:
df_silver_disputes_transactions = dedup(
    df_silver_disputes_transactions,
    dedup_cols=["dispute_id"],
    order_cols=["update_time", "dt", "hour", "elton_created_at"],
    logger=logger
)

2026-04-08 22:18:03 | INFO     | ETL | utils.py:190 | Starting deduplication
2026-04-08 22:18:03 | INFO     | ETL | utils.py:191 | Dedup columns: ['dispute_id']
2026-04-08 22:18:03 | INFO     | ETL | utils.py:192 | Order columns: ['update_time', 'dt', 'hour', 'elton_created_at']
2026-04-08 22:18:03 | INFO     | ETL | utils.py:210 | Order direction (desc): [True, True, True, True]
2026-04-08 22:18:03 | INFO     | ETL | utils.py:211 | Nulls last: True
2026-04-08 22:18:03 | INFO     | ETL | utils.py:218 | Input row count: 254
2026-04-08 22:18:05 | INFO     | ETL | utils.py:253 | Output row count after dedup: 82
2026-04-08 22:18:05 | INFO     | ETL | utils.py:254 | Removed duplicate rows: 172
2026-04-08 22:18:05 | INFO     | ETL | utils.py:255 | Deduplication completed


### Extract Data

In [8]:
disputed_txn_schema = StructType([
    StructField("buyer_transaction_id", StringType()),
    StructField("seller_transaction_id", StringType()),
    StructField("create_time", StringType()),
    StructField("transaction_status", StringType()),
    StructField("gross_amount", StructType([
        StructField("currency_code", StringType()),
        StructField("value", StringType())
    ])),
    StructField("custom", StringType()),
    StructField("buyer", StructType([
        StructField("name", StringType())
    ])),
    StructField("seller", StructType([
        StructField("email", StringType()),
        StructField("merchant_id", StringType()),
        StructField("name", StringType())
    ])),
    StructField("seller_protection_eligible", StringType()),
    StructField("seller_protection_type", StringType()),
])

# Parse JSON -> struct (avoid multiple parsing)
df_parsed = df_silver_disputes_transactions.withColumn(
    "disputed_txn",
    F.from_json(F.col("disputed_transactions"), ArrayType(disputed_txn_schema))
)

# Explode array -> each row = 1 item
df_exploded = df_parsed.withColumn(
    "txn",
    F.explode_outer(F.col("disputed_txn"))
)

# Preview result
df_exploded.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
df_silver_disputes_transactions_final = df_exploded.select(
    "dispute_id",

    # Standardize timestamp
    parse_timestamp(F.col("create_time")).alias("create_time"),
    parse_timestamp(F.col("update_time")).alias("update_time"),

    # buyer_transaction_id: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("txn.buyer_transaction_id")) == "", None)
         .otherwise(F.trim(F.col("txn.buyer_transaction_id"))),
        F.lit("Unknown")
    ).alias("buyer_transaction_id"),

    # seller_transaction_id: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("txn.seller_transaction_id")) == "", None)
         .otherwise(F.trim(F.col("txn.seller_transaction_id"))),
        F.lit("Unknown")
    ).alias("seller_transaction_id"),

    # transaction_status: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("txn.transaction_status")) == "", None)
         .otherwise(F.trim(F.col("txn.transaction_status"))),
        F.lit("Unknown")
    ).alias("transaction_status"),

    # gross_amount: cast to decimal
    F.col("txn.gross_amount.value").cast("decimal(18,2)").alias("gross_amount"),

    # currency_code: trim -> blank -> NULL -> "USD"
    F.coalesce(
        F.when(F.trim(F.col("txn.gross_amount.currency_code")) == "", None)
         .otherwise(F.trim(F.col("txn.gross_amount.currency_code"))),
        F.lit("USD")
    ).alias("currency_code"),

    # custom: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("txn.custom")) == "", None)
         .otherwise(F.trim(F.col("txn.custom"))),
        F.lit("Unknown")
    ).alias("custom"),

    # buyer_name: trim -> blank -> NULL -> "USD"
    F.coalesce(
        F.when(F.trim(F.col("txn.buyer.name")) == "", None)
         .otherwise(F.trim(F.col("txn.buyer.name"))),
        F.lit("Unknown")
    ).alias("buyer_name"),

    # seller_email: trim -> blank -> NULL -> "USD"
    F.coalesce(
        F.when(F.trim(F.col("txn.seller.email")) == "", None)
         .otherwise(F.trim(F.col("txn.seller.email"))),
        F.lit("Unknown")
    ).alias("seller_email"),

    # seller_merchant_id: trim -> blank -> NULL -> "USD"
    F.coalesce(
        F.when(F.trim(F.col("txn.seller.merchant_id")) == "", None)
         .otherwise(F.trim(F.col("txn.seller.merchant_id"))),
        F.lit("Unknown")
    ).alias("seller_merchant_id"),

    # seller_name: trim -> blank -> NULL -> "USD"
    F.coalesce(
        F.when(F.trim(F.col("txn.seller.name")) == "", None)
         .otherwise(F.trim(F.col("txn.seller.name"))),
        F.lit("Unknown")
    ).alias("seller_name"),

    # seller_protection_eligible: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("txn.seller_protection_eligible")) == "", None)
         .otherwise(F.trim(F.col("txn.seller_protection_eligible"))),
        F.lit("Unknown")
    ).alias("seller_protection_eligible"),

    # seller_protection_type: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("txn.seller_protection_type")) == "", None)
         .otherwise(F.trim(F.col("txn.seller_protection_type"))),
        F.lit("Unknown")
    ).alias("seller_protection_type"),

    # Metadata timestamps
    parse_timestamp(F.col("elton_created_at")).alias("elton_created_at"),
    parse_timestamp2(F.col("dt")).cast("date").alias("dt"),
    F.col("hour").cast("int").alias("hour")
) \
.filter(F.col("dispute_id").isNotNull()) \
.withColumn("process_timestamp", F.date_trunc("second", F.current_timestamp()))

# Preview result
df_silver_disputes_transactions_final.show(n=10, truncate=False)

+------------------+-------------------+-------------------+--------------------+---------------------+------------------+------------+-------------+--------+-------------------+---------------------+------------------+------------+--------------------------+--------------------------+-------------------+----------+----+-------------------+
|dispute_id        |create_time        |update_time        |buyer_transaction_id|seller_transaction_id|transaction_status|gross_amount|currency_code|custom  |buyer_name         |seller_email         |seller_merchant_id|seller_name |seller_protection_eligible|seller_protection_type    |elton_created_at   |dt        |hour|process_timestamp  |
+------------------+-------------------+-------------------+--------------------+---------------------+------------------+------------+-------------+--------+-------------------+---------------------+------------------+------------+--------------------------+--------------------------+-------------------+--------

### Transformed data to Silver Layer

In [10]:
if not spark.catalog.tableExists(SILVER_PATH_DISPUTED_PP02_DISPUTES_TRANSACTIONS):
    logger.info("Silver disputed pp02 disputes transactions table not found. Creating new table...")
    df_silver_disputes_transactions_final.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("append") \
                   .saveAsTable(SILVER_PATH_DISPUTED_PP02_DISPUTES_TRANSACTIONS)
    logger.info("Silver disputed pp02 disputes transactions table created successfully")
else:
    logger.info("Silver disputed pp02 disputes transactions table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver_disputes_transactions_final, key_cols=["dispute_id"],
           table=SILVER_TABLE_DISPUTED_PP02_DISPUTES_TRANSACTIONS, cdc="update_time",
           name_catalog=app.catalog_name, name_schema=SCHEMA_SILVER, logger=logger)
    logger.info("Upsert completed successfully")

2026-04-08 21:51:03 | INFO     | ETL | 3358271257.py:10 | Silver disputed pp02 disputes transactions table exists. Performing upsert...
2026-04-08 21:51:03 | INFO     | ETL | utils.py:349 | Starting UPSERT into paypal_analytic.silver.disputed_pp02_disputed_transactions
2026-04-08 21:51:28 | INFO     | ETL | utils.py:379 | UPSERT completed successfully: paypal_analytic.silver.disputed_pp02_disputed_transactions
2026-04-08 21:51:28 | INFO     | ETL | 3358271257.py:14 | Upsert completed successfully


In [11]:
app.stop()

2026-04-08 21:51:28 | INFO     | ETL | platform_app.py:259 | Stopping Spark session...
2026-04-08 21:51:29 | INFO     | ETL | platform_app.py:261 | Spark stopped.
